In [1]:
import os

In [2]:
%pwd

'/Users/justicesmacboookair/Documents/Data-science/SLR-ENP/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/justicesmacboookair/Documents/Data-science/SLR-ENP'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from src.SLR_ENP.constants import *
from src.SLR_ENP.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [8]:
#update components
import os
import urllib.request as request
import zipfile
from src.SLR_ENP.logging import logger
from src.SLR_ENP.utils.common import get_size

In [9]:
class DataIngestion:
    def __init__(self,config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename= self.config.local_data_file
            )
            logger.info(f"{filename} Download! with the following info: \n{headers}")
        else:
            logger.info(f"file already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [10]:
#pipeline
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2025-04-18 21:45:46,421: INFO:common: yaml file: /Users/justicesmacboookair/Documents/Data-science/SLR-ENP/config/config.yaml loaded successfully]
[2025-04-18 21:45:46,423: INFO:common: yaml file: /Users/justicesmacboookair/Documents/Data-science/SLR-ENP/params.yaml loaded successfully]
[2025-04-18 21:45:46,425: INFO:common: yaml file: schema.yaml loaded successfully]
[2025-04-18 21:45:46,426: INFO:common: created directory at: artifacts]
[2025-04-18 21:45:46,426: INFO:common: created directory at: artifacts/data_ingestion]
[2025-04-18 21:45:48,130: INFO:2188243028: artifacts/data_ingestion/data.zip Download! with the following info: 
Connection: close
Content-Length: 302594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "3a5b1a62dbfa5b2ac4fb85a906cb5992ade02f469b8bc202bacbc8a817e2e58b"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-P